# 2.7 — which architecture, on the settled pipeline

Phase 3. Everything that is not the model was settled in phase 2 and the focal series;
this notebook changes **only the model** and holds the rest fixed:

```
64x64 letterbox, one_hot (3 channels)   rotation p=0.5, +/-180 deg
inverse-sqrt weighted sampler           cross-entropy
AdamW lr 7e-4, wd 1e-4, no schedule     batch 256, fp16 AMP
50 epochs, early stop on validation macro-F1, patience 10
```

That is the configuration that scored **0.8800** on `baseline_cnn`.

| arm | params | measured epoch cost |
|---|---|---|
| `v27-baseline_cnn` | 157k | 13s — the reference, rerun here |
| `v27-convnext_style` | 414k | not measured |
| `v27-densenet_style` | 304k | 164s, 3.94 GiB |
| `v27-inception_style` | 799k | not measured |
| `v27-resnet_style` | 2.83M | not measured |

**This is a long session.** densenet alone is ~2.3 h at 50 epochs; four heavy arms could be
8–12 h on a T4. The arms run cheapest-first, the results file is rewritten after every arm,
and a rerun skips whatever is already in it — so a dropped runtime costs one arm, not the set.

## What is already settled (do not re-test)

| finding | evidence |
|---|---|
| Free-angle rotation is the only intervention that cleared its interval | 0.8800 vs 0.8173; `Scratch` 0.376 → 0.732 |
| Composing rotation with `dihedral8` does not beat rotation alone | 0.8677 vs 0.8800 |
| Imbalance handling is not the binding constraint | every phase-2 arm inside ±0.013 |
| Focal loss makes no measurable difference | best arm +0.0042 against a 0.010 noise floor |
| CBAM adds nothing | 0.8158 vs 0.8173 |
| Grayscale is worse than one-hot | 0.8042 vs 0.8173 |
| 224x224 does not pay | 0.8321 for 6.4x the compute |
| `baseline_v2` is worse than `baseline_cnn` once rotation is on | ~0.80 vs ~0.87 — dropped |

## The bar

The noise floor is **0.010**, measured directly: the identical config scored 0.8800 and
0.8696. An architecture beats the baseline only if its bootstrap CI lower bound sits above
the baseline's point estimate in *this* session. Anything closer is a tie, and a tie is won
by the cheaper model.

## 1. Setup

Same bootstrap as the phase-2 notebook; every step is a no-op if already done.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

## 2. W&B

`wandb login` in the terminal is the reliable path — Colab Secrets time out when the runtime
is driven from VS Code. `~/.netrc` is read by both this kernel and any terminal process.

In [ ]:
USE_WANDB = True
WANDB_PROJECT = "wm811k-wafer-defects"

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

    if not wandb.api.api_key:
        USE_WANDB = False
        print("  not authenticated -- run `wandb login` in the terminal, then rerun")
    else:
        print(f"  wandb ready, project {WANDB_PROJECT!r}")

## 3. Run the arms

Sequentially, on one loaded copy of the source table. `transform_device=cuda` moves the
one-hot encoding and the rotation onto the GPU, which matters most for the cheap arms;
the heavy ones are GPU-bound already (densenet measured 96% util) and will not speed up.

Resumable in two independent ways:

* **Between arms** — a finished arm is appended to `results.csv` immediately, and rerunning
  this cell skips every arm already in that file. Set `RERUN = True` to force all of them.
* **Within an arm** — checkpoints go to Drive and `checkpoint.resume: auto` picks up the
  newest epoch, so a runtime that dies mid-arm resumes rather than restarts.

In [ ]:
import time

import pandas as pd

from fdl_project.config.loader import load_experiment_config
from fdl_project.config.registry import build_model
from fdl_project.data.datasets import load_wm811k_dataframe
from fdl_project.models.baseline_cnn import count_trainable_parameters
from fdl_project.training.runner import run_experiment

SERIES = "v27_models"
RERUN = False                     # True re-trains arms already in results.csv
BASELINE = "v27-baseline_cnn"     # the arm everything is measured against

CONFIGS = sorted((REPO / "configs/train" / SERIES).glob("*.yaml"))
assert CONFIGS, f"no configs in configs/train/{SERIES}"

OUTPUT = REPO / "output" / SERIES
OUTPUT.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUTPUT / "results.csv"

OVERRIDES = ["data.transform_device=cuda"]
if HAS_DRIVE:
    OVERRIDES.append(f"checkpoint.directory={CHECKPOINTS}")
if USE_WANDB:
    OVERRIDES += ["logging.wandb.enabled=true",
                  f"logging.wandb.project={WANDB_PROJECT}",
                  f"logging.wandb.tags=[{SERIES},phase3]"]

results = []
if RESULTS_CSV.exists() and not RERUN:
    results = pd.read_csv(RESULTS_CSV).to_dict("records")
    print(f"resuming: {len(results)} arm(s) already done -- "
          + ", ".join(str(r["run"]) for r in results))

done = {r["run"] for r in results}
dataframe = load_wm811k_dataframe(DATASET)

for path in CONFIGS:
    config = load_experiment_config(path, overrides=OVERRIDES)
    # The bug that silently invalidated the first focal pass: a config in a
    # subdirectory stopped inheriting defaults.yaml and trained at batch 512 /
    # lr 1e-3 / 40 epochs. Fail here rather than produce a wrong number.
    assert config.trainer.max_epochs == 50, "defaults.yaml was not inherited"
    assert config.trainer.batch_size == 256, "defaults.yaml was not inherited"
    assert config.data.augmentation.name == "rotation", "settled pipeline is rotation"
    assert config.data.preprocessing.encoding == "one_hot", "settled pipeline is one_hot"

    if config.name in done:
        print(f"skip  {config.name}  (already in results.csv)")
        continue

    parameters = count_trainable_parameters(
        build_model(config.model.name, **config.model.kwargs)
    )
    print(f"\n=== {config.name}  ({config.model.name}, {parameters:,} parameters)")
    started = time.monotonic()
    result = run_experiment(config, overwrite=True, dataframe=dataframe)
    macro = result.bootstrap.aggregate.set_index("metric").loc["macro_f1"]
    results.append({
        "run": config.name,
        "model": config.model.name,
        "parameters": parameters,
        "macro_f1": round(float(macro.point_estimate), 4),
        "ci_lower": round(float(macro.ci_lower), 4),
        "ci_upper": round(float(macro.ci_upper), 4),
        "best_epoch": result.fit.best_epoch,
        "epochs": len(result.fit.history),
        "minutes": round((time.monotonic() - started) / 60, 1),
    })
    row = results[-1]
    # Written after every arm: a runtime that dies later keeps this one.
    pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)
    if HAS_DRIVE:
        shutil.copy2(RESULTS_CSV, DRIVE / f"{SERIES}_results.csv")
    flag = "  <-- still improving at the cap" if row["best_epoch"] >= row["epochs"] - 2 else ""
    print(f"    macro-F1 {macro.point_estimate:.4f} "
          f"[{macro.ci_lower:.4f}, {macro.ci_upper:.4f}]  "
          f"best {row['best_epoch']}/{row['epochs']}  {row['minutes']:.1f} min{flag}")

print(f"\n{len(results)}/{len(CONFIGS)} arms complete -> {RESULTS_CSV}")

## 4. Read the result

`beats_baseline` is the only column that ranks anything: the arm's CI lower bound has to sit
above the baseline's point estimate. With 30 `Near-full` wafers in validation, point
estimates alone are not a ranking.

`per_100k_params` is the tie-breaker. On a nine-class problem with 121k training wafers, an
architecture that needs 18x the parameters to land inside the noise floor of a 157k CNN has
not made a case for itself.

In [ ]:
frame = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
assert BASELINE in set(frame["run"]), f"{BASELINE} has not run yet -- nothing to compare to"

baseline_f1 = float(frame.loc[frame["run"] == BASELINE, "macro_f1"].squeeze())
frame["vs_baseline"] = (frame["macro_f1"] - baseline_f1).round(4)
frame["beats_baseline"] = frame["ci_lower"] > baseline_f1
# A best epoch at the cap means the run was still improving when it stopped:
# that number is a floor, not a result. The bigger models are the likely case.
frame["truncated"] = frame["best_epoch"] >= frame["epochs"] - 2
frame["per_100k_params"] = (frame["macro_f1"] / (frame["parameters"] / 100_000)).round(4)

pd.set_option("display.width", 240)
display(frame)

print(f"\nBaseline {baseline_f1:.4f}  (phase 2 scored the same config at 0.8800, "
      f"a rerun at 0.8696 -- the 0.010 spread is the noise floor)")

if frame["truncated"].any():
    print("\nTruncated (still improving at 50 epochs):",
          ", ".join(frame.loc[frame["truncated"], "run"]))
    print("Those are lower bounds. Raise trainer.max_epochs for them before concluding.")

winners = frame[frame["beats_baseline"] & (frame["run"] != BASELINE)]
if winners.empty:
    print("\nNo architecture clears the baseline's point estimate. The finding is that "
          "on 64x64 wafer maps a 157k CNN is enough, and augmentation -- not capacity -- "
          "was the binding constraint. Report it with these numbers.")
else:
    print("\nClears the baseline:", ", ".join(winners["run"]))
    print("Confirm with three seeds before it goes in the report; one run is one draw.")

## 5. Per-class, where the difference would actually show

Macro-F1 hides *which* class moved. The baseline's weak classes are `Scratch` (0.73 with
rotation, up from 0.38 without) and `Donut`/`Near-full`, which have 111 and 30 validation
wafers. If a bigger architecture pays at all, it pays there.

In [ ]:
per_class = {}
for run in frame["run"]:
    path = REPO / "output/runs" / run / "per_class_metrics.csv"
    if path.exists():
        table = pd.read_csv(path).set_index("class_name")
        per_class[run] = table["f1"]

if per_class:
    comparison = pd.DataFrame(per_class)
    if BASELINE in comparison:
        others = [c for c in comparison.columns if c != BASELINE]
        comparison = comparison[[BASELINE, *others]]
        for column in others:
            comparison[f"{column}_delta"] = (comparison[column] - comparison[BASELINE]).round(3)
    display(comparison.round(3))
else:
    print("No per_class_metrics.csv found -- run section 3 first.")

## 6. What follows

* **If nothing beats the baseline** — that is the result, and it is a real one: augmentation
  was the binding constraint, not capacity. The report then compares architectures on cost
  and stability rather than on a winner, and the final test evaluation runs on
  `baseline_cnn`.
* **If something does** — three seeds on it before it is believed, exactly as with the
  control. One run on a validation set this imbalanced is one draw.

Either way the pretrained backbones (`resnet18`, `efficientnet_b0`, `vit_b_16`) are the
separate question, and they need 224x224 — which phase 2 already measured as not paying for
`baseline_cnn` (0.8321 for 6.4x the compute). Decide that after this.